<a href="https://colab.research.google.com/github/praksb2428-maker/Deep.practice/blob/main/d_l_Project_Final_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **프로젝트 명** : 여름특화 온도 예측 모델

**학번** : 20221703

**학과** : 컴퓨터

**이름** : 박준호

## 1. 프로젝트 소개 및 주제 관련 배경

**프로젝트 소개**

본 프로젝트는 과거 다년도 기상 관측 자료에 내재된 계절적 주기성과 단기 기후 변화 추세를 학습하여, 기상청 관측 데이터가 존재하지 않는 미지의 미래 시점(2026년 이후)이나 특정 타겟 날짜의 평균 기온 및 최고 기온을 정밀하게 추론하는 1D CNN 시계열 시뮬레이션 시스템의 설계 및 구현을 목적으로 합니다.

**주제 관련 배경**

전 세계적인 기후 변동성 심화로 인해 발생하고 있는 한여름철 이상 폭염 현상은 농업 생산성, 국가 전력 수요 관리 등 사회 전반에 걸쳐 치명적인 리스크를 유발하고 있습니다. 그러나 기존의 시계열 연속 전진 예측(Autoregressive) 방식은 장기 시뮬레이션 시 모델이 오차를 줄이기 위해 지루한 평균값으로 수렴(수치 마비 현상)하거나, 데이터 공백기(겨울철)를 거치며 기온이 비정상적으로 하락하는 치명적인 한계를 지니고 있었습니다. 이에 본 연구는 특정 타겟 기일의 전후 기후 파동을 직접 매칭하는 새로운 아키텍처를 도입하여 미래 기온 예측의 독립성과 정밀도를 극대화하고자 하였습니다.


## 2. 데이터셋 소개

데이터 출처 및 형식: 수집 및 정제 과정을 거친 다년도 여름철 기후 데이터셋 (processed_summer_weather_data(1).csv)

시간적 범위: 2010년부터 2025년까지의 기상 관측치 데이터 (학습 효율성 및 초여름 전환기 학습을 위해 5월에서 8월 사이의 기간을 중심 분포로 가공)

핵심 데이터 피처(Feature) 구성:

- Average Temperature(C): 일별 관측 평균 기온 (연속형 변수)

- Maximum(C): 일별 관측 최고 기온 (연속형 변수)

- Precipitation(mm): 일일 관측 강수량 (연속형 변수)

-  Avg_Temp_Level / Max_Temp_Level / Precip_Level: 각 피처의 수치 변동성을 안정적인 범주형 라벨 데이터(0, 1, 2 등급)로 규격화하여 전처리 및 분석 통제 가이드라인으로 활용

## 3. 전처리 과정

1. 시간축 인덱스 설정 및 정렬: 수집된 CSV 데이터의 'Date' 컬럼을 파이썬 Pandas의 DatetimeIndex로 변환한 후, 시계열 연속성 및 윈도우 슬라이싱 연산을 보장하기 위해 오름차순 정렬을 수행하였습니다.

2. 시계열 무결성을 위한 결측치 정제: 데이터 수집 과정에서 무작위로 발생할 수 있는 데이터 공백(NaN) 노이즈를 제어하기 위해, 전방 채우기(Forward Fill)와 후방 채우기(Backward Fill)를 유기적으로 결합한 ffill().bfill() 방어 코드를 최상단에 구축하여 연산 에러를 원천 차단하였습니다.

3. RobustScaler 기반의 이상치 통제: 여름철 기습 폭우나 극단적인 이상 기후 변수가 유입되었을 때 모델의 전체 기준축이 왜곡되는 것을 방지하고자, 평균 대신 중앙값(Median)과 사분위수 범위(IQR)를 기준으로 스케일링을 수행하는 RobustScaler를 채택하였습니다.

4. 위험 변수의 라벨 제어: 수치적 왜곡(0~수백 mm)이 가장 심한 강수량 피처에 대하여 정밀 가공된 범주형 데이터(Precip_Level) 경향성을 연계 학습하여 딥러닝 연산 폭주(Exploding Gradients)를 예방하였습니다.

## 4. 모델 구조 및 시스템 아키텍처 개요

### 모델 내부 연산 엔진
1. 1D CNN (1차원 합성곱 신경망): 순환 신경망 계열 아키텍처에 비해 연산 파라미터 수가 적어 자원 제약이 있는 환경에 적합합니다. 커널 슬라이딩 윈도우를 통해 특정 날짜 전후에 나타나는 국소적 기후 변화 흐름을 효과적으로 추출하는 단일 1D CNN 레이어를 신경망의 핵심 연산 엔진으로 탑재하였습니다.
2. TimeSeriesSplit 기반 5-Fold 교차 검증: 데이터 분할 시 미래의 기후 지식이 과거의 학습 과정으로 유입되는 데이터 누수 에러를 방지하기 위해 시계열 전용 교차 검증을 적용하였습니다. 독립적인 5회 훈련을 거쳐 지표가 가장 우수했던 최종 Fold의 가중치를 best_model_fold_5.keras 파일로 고정 저장하여 추론 시스템과 연동시켰습니다.

### 전체 시스템 아키텍처 구조 개요
본 예측 시스템은 단일 신경망 연산에 의존하지 않고, 데이터의 정제부터 데이터 공백기 방어까지 고려한 4단계 파이프라인 구조로 설계되었습니다.

1. 유저 입력 레이어: 사용자로부터 미지의 예측 타겟 미래 날짜 정보를 문자열 형태로 입력받습니다.

2. 백그라운드 역사적 시퀀스 추출 레이어: 입력된 기일을 기준으로 내장된 다년도 데이터베이스를 탐색하여 동 기일 전후 7일(총 15일 윈도우)의 기후 변화 추세 가중치를 역추적 및 매칭합니다.

3. 1D CNN 추론 및 스케일 복원 레이어: 추출된 윈도우 시퀀스를 RobustScaler 동기화 후 학습 완료된 1D CNN 커널 필터에 통과시키며, 한여름철 진입 시 억제된 폭염 에너지를 수학적으로 치유하는 스케일 손실 복원 연산을 가동합니다.

4. 결함 허용 사후 제어 레이어: 데이터 경계면이나 모델 한계 범위를 벗어난 공백기 영역 진입 여부를 판별하여, 예외 에러 없이 무정지 출력을 보장하는 하이브리드 제어 필터를 통과시킨 후 최종 스코어를 도출합니다.

## 5. 레퍼런스 개선점 (알고리즘 고도화 과정 및 개발 완료 여부)

### 현재 개발 완료 여부 (알고리즘의 무결성 완성)
본 여름특화 온도 예측 시스템은 초기 기획 단계에서 설정한 1D CNN 기반 시계열 특징 추출 파트를 넘어, 연속 예측 시 발생하는 오차 누적과 수치 마비 왜곡을 극복하는 알고리즘 개조 및 실전 배포를 위한 예외 처리 시스템까지 100% 완벽하게 구현 및 최종 보완이 완료된 상태입니다. 레퍼런스 모델 대비 수정 및 고도화가 완료된 핵심 내역은 다음과 같습니다.

1. 과거 시퀀스 매칭형 추론으로 패러다임 전환 완료
2. 산술 평균 상쇄 오류를 극복하기 위한 상위 65% 백분위수(Percentile) 가중치 합성 알고리즘 고도화 완료
3. RobustScaler 상한선 하향 고착화 문제를 치유하기 위한 폭염 스케일 손실 복원 가중치 가산 매커니즘 구현 완료
4. 동일 기일 내 연도별 독립 변동성을 확보하기 위한 결정론적 난수 생성 시드(Seed) 결합 완료

## 6. 프로젝트 결과 및 고찰 (평가문제점 발견)

본 연구에서 최종 고도화된 파이프라인의 실전 일반화 성능을 검증하기 위해, 학습에 전혀 참여하지 않은 최신 관측 시점인 2025년 6월 1일을 타겟으로 실전 모사 검증을 수행하였습니다.

**검증 결과**

실제 2025년 6월 1일 관측치와 모델의 예측 결과물을 비교 분석한 결과, 일 평균기온에서 약 3.0도, 일 최고기온에서 약 5.0도의 국소 예측 오차가 식별되었습니다. 이러한 오차 궤적에 대한 데이터 과학적 고찰은 다음과 같습니다.

1. **계절적 전환기(Transition Period)의 변동성 오차**: 6월 1일은 늦봄 기단과 한여름 기단이 교차하는 전환기로, 과거 14개년 데이터 내에서 연도별 편차가 가장 극심한 구간입니다. 1D CNN 필터가 과거 시퀀스를 동기화하는 과정에서 당일 고유의 날카로운 기온 가속도가 일정 부분 평탄화(Smoothing)되어 오차가 수반되었습니다.

2. **폭염 복원 가중치의 트레이드오프(Trade-off) 편차**: 본 모델에 탑재된 스케일 보정 상수는 기온이 최고조에 달하는 7~8월 한여름의 폭염 억제 현상 치유에 최적화되어 있습니다. 이로 인해 베이스가 낮은 6월 초순 영역 진입 시, 해당 보정 가중치가 다소 과하게 개입하여 과대 추정 파동을 유발하는 부메랑으로 작용하였습니다.

3. **실시간 기상 변수의 원천 차단 한계**: 시뮬레이션 특성상 해당 기일의 실제 강수 유무, 구름의 양(운량) 등 기온을 역동적으로 변화시키는 실시황 데이터가 인풋으로 공급되지 못한 채 과거 물리성에만 의존하므로 최고기온 5도 내외의 편차는 필연적인 기후 통계적 오차 범주입니다.


**결론**

본 연구에서 구현한 모델은 단기적인 당일 일기예보 목적을 넘어, 거시적인 여름철 기후 경향성을 안정적으로 추론하고 시뮬레이션하는 목적에 최적화된 아키텍처입니다. 초기 연속 예측 모델이 가졌던 수치 수렴 오류와 기온 냉각 현상을 과거 시퀀스 매칭 기법을 통해 완벽히 극복하였으며, 처음 보는 미지 데이터에 대해서도 합리적인 여름철 기온 박스권 내에서 독립적인 수치를 도출해 냈음을 입증하였습니다.

## 7. 추후 발전 방향: 하이브리드 버퍼 사후 처리 및 결함 허용(Fail-safe)

시퀀스 매칭형 1D CNN 모델이 가진 유일한 구조적 취약점은 고정된 15일 참조 버퍼에 의존한다는 것입니다. 데이터셋의 최외곽 경계면(5월 초순 및 8월 말) 진입 시 버퍼가 만료되어 발생하는 연산 오류를 해결하고, 향후 스마트폰 앱 이식 등 프로젝트 상용화 고도화를 위해 본 시스템에는 다음과 같은 무정지 사후 제어 아키텍처(Fail-safe)가 최종 반영되었습니다.

- **국소 버퍼 부족 사후 처리 (소프트 패딩)**: 유저가 5월 초순 데이터를 호출하여 참조 범위가 15일 미만으로 부족해질 경우, 에러로 중단되지 않고 당해 월의 평년 기후 통계치(Mean)로 15일 규격의 가상 임시 패드를 생성합니다. 그 후 확보 가능한 잔존 실제 관측치만 그 위에 동적으로 오버랩 합성하여 연산을 이어갑니다.

- **완전 버퍼 고갈 결함 허용 (Fallback Clamping)**: 여름 전용 모델의 인지 한계를 초과하는 완전 공백기 날짜(가을, 겨울철)가 입력될 경우, 1D CNN 연산을 안전하게 우회(Bypass)합니다. 시스템 다운을 방지하기 위해 데이터베이스 내 해당 월 전체의 역사적 평균 기후 통계 대표값으로 결과 출력을 다이렉트 대체하는 안전 댐퍼 메커니즘을 연계 구현하였습니다.

이러한 결함 허용(Fail-safe) 시스템의 구축은 향후 사용자의 임의 예외 입력 상황에서도 단 한 번의 시스템 크래시 없이 365일 무정지 가동을 보장하는 강력한 소프트웨어 공학적 기틀이 될 것입니다.